# CLAMPfull prediction of cell proportions in test samples

For each training fold, CLAMPfull assigns one unique LV to every cell type using the maximum positive Pearson correlation. A dataset–cell-type result is valid only when the selected LV has training `r >= 0.5` in all five folds. Test samples are not used to train CLAMPfull, select LVs, fit calibrations, or determine validity.


💡 **Environment:** `clamp-analyses`

## Libraries

In [ ]:
suppressPackageStartupMessages({
  library(data.table)
  library(ggplot2)
  library(here)
})

## Settings

In [ ]:
DATASETS <- snakemake@params[["datasets"]]
OUT <- here(snakemake@config$paths$production, 'grouped_cv_analysis')
OUT_DIR <- here(snakemake@params[["out_dir"]])
dir.create(OUT_DIR, recursive = TRUE, showWarnings = FALSE)
MIN_TRAIN_LV_COR <- as.numeric(snakemake@config$grouped_cv$min_train_lv_cor)
CLAMP_COLOR <- '#0072B2'

predictions <- fread(file.path(OUT, 'oof_predictions.csv'))
thresholded_metrics <- fread(file.path(OUT, 'thresholded_metrics.csv'))
thresholded_summary <- fread(file.path(OUT, 'thresholded_summary.csv'))

stopifnot(
  'valid_all_folds' %in% names(thresholded_metrics),
  all(predictions$min_train_lv_cor == MIN_TRAIN_LV_COR),
  all(thresholded_metrics$min_train_lv_cor == MIN_TRAIN_LV_COR)
)

## Results

Test metrics are reported only when all five training folds reach `r >= 0.5`. Invalid results remain in the table with missing test metrics.


In [ ]:
results <- thresholded_metrics[, .(
  dataset,
  cell_type,
  valid_all_folds,
  minimum_training_r = minimum_observed_train_lv_cor,
  folds_passing = n_eligible_folds,
  total_folds = n_total_folds,
  n_test = n_total_test,
  test_pearson_r = pearson_r,
  prediction_r_squared = predictive_r2,
  mae
)][order(dataset, cell_type)]

results


## Mean test performance


In [ ]:
summary_row <- thresholded_summary[method == 'CLAMPfull'][1]
mean_test_performance <- data.frame(
  training_threshold = summary_row$min_train_lv_cor,
  valid_dataset_cell_types = summary_row$n_valid,
  invalid_dataset_cell_types = summary_row$n_invalid,
  mean_test_pearson_r = summary_row$mean_pearson_r,
  mean_prediction_r_squared = summary_row$mean_predictive_r2,
  mean_mae = summary_row$mean_mae
)

mean_test_performance


## Training and test results

Blue points are test predictions for valid dataset–cell-type results. Grey crosses show test predictions from invalid results and are excluded from all reported test metrics. The dashed line represents perfect prediction.


In [ ]:
valid_label <- sprintf('Valid: all training folds r >= %.2f', MIN_TRAIN_LV_COR)
invalid_label <- sprintf('Invalid: at least one training fold r < %.2f', MIN_TRAIN_LV_COR)

thresholded_metrics[, panel_label := fifelse(
  valid_all_folds,
  sprintf(
    '%s\nValid: %d/%d training folds\nTest r = %.2f; R-squared = %.2f; MAE = %.3f',
    cell_type, n_eligible_folds, n_total_folds, pearson_r, predictive_r2, mae
  ),
  sprintf(
    '%s\nInvalid: %d/%d training folds reached r >= %.2f\nTest metrics not reported',
    cell_type, n_eligible_folds, n_total_folds, min_train_lv_cor
  )
)]

plot_data <- merge(
  predictions,
  thresholded_metrics[, .(dataset, method, cell_type, valid_all_folds, panel_label)],
  by = c('dataset', 'method', 'cell_type'),
  all.x = TRUE
)
plot_data[, result_status := factor(
  fifelse(valid_all_folds, valid_label, invalid_label),
  levels = c(valid_label, invalid_label)
)]

make_test_prediction_plot <- function(dataset_name) {
  dataset_predictions <- plot_data[dataset == dataset_name]
  dataset_panels <- thresholded_metrics[dataset == dataset_name][order(cell_type), panel_label]
  dataset_predictions[, panel_label := factor(panel_label, levels = dataset_panels)]
  n_panels <- uniqueN(dataset_predictions$panel_label)
  n_columns <- min(4L, max(1L, ceiling(sqrt(n_panels))))
  n_rows <- ceiling(n_panels / n_columns)
  options(repr.plot.width = 13, repr.plot.height = max(4.5, 3.2 * n_rows))

  ggplot(
    dataset_predictions,
    aes(x = observed, y = predicted, color = result_status, shape = result_status)
  ) +
    geom_abline(slope = 1, intercept = 0, color = 'grey55', linetype = 'dashed') +
    geom_point(alpha = 0.72, size = 1.7) +
    facet_wrap(~panel_label, scales = 'free', ncol = n_columns) +
    scale_color_manual(
      values = setNames(c(CLAMP_COLOR, 'grey70'), c(valid_label, invalid_label)),
      drop = FALSE
    ) +
    scale_shape_manual(
      values = setNames(c(16, 4), c(valid_label, invalid_label)),
      drop = FALSE
    ) +
    labs(
      x = 'True cell proportion in test sample',
      y = 'Predicted cell proportion in test sample',
      color = NULL,
      shape = NULL,
      title = paste0(dataset_name, ': test cell-proportion predictions')
    ) +
    theme_bw(base_size = 10) +
    theme(
      panel.grid.minor = element_blank(),
      strip.text = element_text(size = 8),
      legend.position = 'bottom'
    )
}

for (dataset_name in DATASETS) {
  print(make_test_prediction_plot(dataset_name))
}


Each panel contains only cell types valid in all five training folds. Reported values are means of the cell-type-specific metrics, not metrics calculated after pooling the plotted points.

In [ ]:
valid_keys <- thresholded_metrics[
  valid_all_folds == TRUE,
  .(dataset, method, cell_type)
]
valid_test_predictions <- merge(
  predictions, valid_keys,
  by = c('dataset', 'method', 'cell_type')
)

scatter_data <- copy(valid_test_predictions)
scatter_data[, panel_label := dataset]

stopifnot(
  uniqueN(scatter_data$dataset) == 6L,
  uniqueN(scatter_data[, .(dataset, cell_type)]) == 35L
)

plot_statistics <- scatter_data[, {
  fit <- lm(predicted ~ observed)
  fit_summary <- summary(fit)
  list(
    pooled_r2 = fit_summary$r.squared,
    p_value = fit_summary$coefficients['observed', 'Pr(>|t|)']
  )
}, by = panel_label]
setorder(plot_statistics, -pooled_r2)
panel_levels <- plot_statistics$panel_label
scatter_data[, panel_label := factor(panel_label, levels = panel_levels)]
plot_statistics[, panel_label := factor(panel_label, levels = panel_levels)]
plot_statistics[, annotation := fifelse(
  p_value < 0.001,
  sprintf('atop(R^2 == %.2f, italic(p) < 0.001)', pooled_r2),
  sprintf('atop(R^2 == %.2f, italic(p) == %.3f)', pooled_r2, p_value)
)]

options(repr.plot.width = 8, repr.plot.height = 11)
dataset_scatter_plot <- ggplot(
  scatter_data,
  aes(x = observed, y = predicted)
) +
  geom_abline(slope = 1, intercept = 0, color = 'black', linetype = 'dashed', linewidth = 0.55) +
  geom_point(color = CLAMP_COLOR, alpha = 0.24, size = 0.85) +
  geom_text(
    data = plot_statistics,
    aes(x = 0.03, y = 0.97, label = annotation),
    inherit.aes = FALSE, hjust = 0, vjust = 1, size = 2.8, parse = TRUE
  ) +
  facet_wrap(~panel_label, ncol = 2) +
  coord_fixed(xlim = c(0, 1), ylim = c(0, 1), expand = FALSE) +
  scale_x_continuous(
    breaks = seq(0, 1, 0.25),
    labels = c('0', '.25', '.5', '.75', '1')
  ) +
  scale_y_continuous(breaks = seq(0, 1, 0.25)) +
  labs(
    x = 'True cell proportion in test sample',
    y = 'Predicted cell proportion in test sample'
  ) +
  theme_classic(base_size = 9) +
  theme(
    strip.background = element_rect(fill = 'grey94', color = 'black', linewidth = 0.4),
    strip.text = element_text(size = 8),
    panel.spacing.x = grid::unit(0.5, 'lines')
  )
dataset_scatter_plot